In [1]:
%load_ext cudf.pandas

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
# %%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
# from tpch import load_part, load_partsupp, load_supplier, q16
import pandas as pd
DATA_ROOT = "/home/colinc/code/tpch/data/tpch_15"#"/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}




In [ ]:

### cell 0 ###

def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [ ]:

### cell 1 ###

def load_partsupp(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/partsupp.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
partsupp = load_partsupp(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
### cell 2 ###

def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:

# %%time
# ### cell 3 ###

# ### cell 3 (optimized for cudf)
# # restrict part to only the columns and rows we need
# sizes = [49, 14, 23, 45, 19, 3, 36, 9]
# part_filtered = (
#     part
#     [(part.P_BRAND != "Brand#45")
#      & (~part.P_TYPE.str.contains(r"^MEDIUM POLISHED"))
#      & part.P_SIZE.isin(sizes)]
#     [["P_PARTKEY", "P_BRAND", "P_TYPE", "P_SIZE"]]
# )

# # restrict partsupp to the columns we need
# partsupp_filtered = partsupp[["PS_PARTKEY", "PS_SUPPKEY"]]

# # inner‐join part and partsupp
# total = (
#     part_filtered
#     .merge(partsupp_filtered,
#            left_on="P_PARTKEY", right_on="PS_PARTKEY",
#            how="inner")[["P_BRAND", "P_TYPE", "P_SIZE", "PS_SUPPKEY"]]
# )

# # build a deduplicated list of suppliers to exclude
# supplier_keys = (
#     supplier
#     [supplier.S_COMMENT.str.contains(r"Customer(\S|\s)*Complaints")]
#     ["S_SUPPKEY"].drop_duplicates()
# )

# # filter out any PS_SUPPKEY that appears in that supplier list
# total = total[~total.PS_SUPPKEY.isin(supplier_keys)]

# # group, count unique suppliers, rename, and sort
# total = (
#     total
#     .groupby(["P_BRAND", "P_TYPE", "P_SIZE"],
#               as_index=False, sort=False)
#     .agg({"PS_SUPPKEY": "nunique"})
#     .rename(columns={"PS_SUPPKEY": "SUPPLIER_CNT"})
#     .sort_values(
#         by=["SUPPLIER_CNT", "P_BRAND", "P_TYPE", "P_SIZE"],
#         ascending=[False, True, True, True]
#     )
# )


In [ ]:
%%time
#original
part_filtered = part[
    (part["P_BRAND"] != "Brand#45")
    & (~part["P_TYPE"].str.contains("^MEDIUM POLISHED"))
    & part["P_SIZE"].isin([49, 14, 23, 45, 19, 3, 36, 9])
]
part_filtered = part_filtered.loc[:, ["P_PARTKEY", "P_BRAND", "P_TYPE", "P_SIZE"]]
partsupp_filtered = partsupp.loc[:, ["PS_PARTKEY", "PS_SUPPKEY"]]
total = part_filtered.merge(
    partsupp_filtered, left_on="P_PARTKEY", right_on="PS_PARTKEY", how="inner"
)
total = total.loc[:, ["P_BRAND", "P_TYPE", "P_SIZE", "PS_SUPPKEY"]]
supplier_filtered = supplier[
    supplier["S_COMMENT"].str.contains(r"Customer(\S|\s)*Complaints")
]
supplier_filtered = supplier_filtered.loc[:, ["S_SUPPKEY"]].drop_duplicates()
# left merge to select only PS_SUPPKEY values not in supplier_filtered
total = total.merge(
    supplier_filtered, left_on="PS_SUPPKEY", right_on="S_SUPPKEY", how="left"
)
total = total[total["S_SUPPKEY"].isna()]
total = total.loc[:, ["P_BRAND", "P_TYPE", "P_SIZE", "PS_SUPPKEY"]]
total = total.groupby(["P_BRAND", "P_TYPE", "P_SIZE"], as_index=False, sort=False)[
    "PS_SUPPKEY"
].nunique()
total.columns = ["P_BRAND", "P_TYPE", "P_SIZE", "SUPPLIER_CNT"]
total = total.sort_values(
    by=["SUPPLIER_CNT", "P_BRAND", "P_TYPE", "P_SIZE"],
    ascending=[False, True, True, True],
)


In [ ]:
429

In [8]:
### cell 4 ###

total.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 27840 entries, 4416 to 25589
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   P_BRAND       27840 non-null  object
 1   P_TYPE        27840 non-null  object
 2   P_SIZE        27840 non-null  int64
 3   SUPPLIER_CNT  27840 non-null  int32
dtypes: int32(1), int64(1), object(2)
memory usage: 1.5+ MB
